# Setup Infrastructure

This notebook provisions the Azure infrastructure required for training and deploying SLMs using Terraform.

## What This Notebook Does

1. **Verifies Prerequisites**: Ensures Azure CLI and Terraform are installed and configured
2. **Configures Terraform**: Reviews and validates Terraform variable files for your environment
3. **Provisions Resources**: Creates Azure ML workspace, storage account, container registry, and networking
4. **Validates Deployment**: Confirms all resources are created successfully and accessible

## Why These Steps Matter

- **Azure ML Workspace**: Central hub for managing training jobs, datasets, models, and compute
- **Storage Account**: Stores training data, model checkpoints, and artifacts with high availability
- **Container Registry**: Hosts optimized Docker images for deployment to embedded devices
- **Resource Group**: Logical container that groups all related Azure resources for easy management

## Prerequisites

Before running this notebook:
- Azure CLI installed and authenticated (`az login`)
- Terraform installed (v1.0+)
- Valid Azure subscription with permissions to create resources
- Reviewed and updated `.env` file with your subscription details

## Expected Duration

~10-15 minutes for full infrastructure provisioning

## 1. Verify Prerequisites

**Why verify prerequisites?** We need to confirm that the required CLI tools are installed and authenticated before attempting infrastructure provisioning. This prevents partial deployments and unclear error messages.

In [ ]:
# Check Azure CLI
!az --version | head -1

# Check current Azure account
!az account show --query '{Name:name, SubscriptionId:id, TenantId:tenantId}' -o table

In [ ]:
# Check Terraform version
!terraform version

## 2. Configure Terraform Variables

Update the `dev.tfvars` or `prod.tfvars` file with your desired configuration.

In [ ]:
# Display current dev configuration
!cat ../infra/terraform/environments/dev.tfvars

### Important Configuration Notes

**Storage Account Names:**
- Must be globally unique across all Azure
- 3-24 characters, lowercase letters and numbers only
- Update `storage_account_name` in tfvars if name is taken

**Container Registry Names:**
- Must be globally unique
- 5-50 characters, alphanumeric only
- Update `acr_name` in tfvars if name is taken

## 3. Initialize Terraform

**Why initialize Terraform?** Terraform initialization downloads the required provider plugins (Azure, random, etc.) and sets up the backend for state management. This only needs to be run once per workspace, or when provider versions change.

In [ ]:
# Change to Terraform directory
import os
os.chdir('../infra/terraform')
!pwd

In [ ]:
# Initialize Terraform (download providers)
!terraform init

## 4. Validate Terraform Configuration

In [ ]:
# Validate syntax
!terraform validate

In [ ]:
# Format Terraform files
!terraform fmt -recursive

## 5. Plan Infrastructure Changes

In [ ]:
# Preview what will be created (dev environment)
!terraform plan -var-file=environments/dev.tfvars -out=tfplan

**Review the plan output above:**
- Ensure resource names are correct
- Verify resource group, location, and SKUs
- Check that no existing resources will be destroyed
- Confirm costs align with expectations

## 6. Apply Infrastructure Changes

⚠️ **WARNING:** This will create real Azure resources that may incur costs!

In [ ]:
# Apply the plan
!terraform apply tfplan

## 7. Verify Resources in Azure Portal

After successful deployment:
1. Go to [Azure Portal](https://portal.azure.com)
2. Navigate to your resource group
3. Verify these resources exist:
   - Azure ML Workspace
   - Storage Account
   - Container Registry
   - (Compute cluster will be created separately)

## 8. Get Output Values

In [ ]:
# Display all outputs
!terraform output

In [ ]:
# Get specific outputs for use in other notebooks
import subprocess
import json

def get_terraform_output(name):
    result = subprocess.run(
        ['terraform', 'output', '-json', name],
        capture_output=True,
        text=True
    )
    return json.loads(result.stdout)

# Store important values
workspace_name = get_terraform_output('workspace_name')
storage_account = get_terraform_output('storage_account_name')
acr_name = get_terraform_output('acr_name')

print(f"Workspace: {workspace_name}")
print(f"Storage: {storage_account}")
print(f"ACR: {acr_name}")

## 9. Test Azure ML Connection

In [ ]:
# Test connection to Azure ML workspace
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from src.utils.config import load_config
from src.utils.azure_auth import get_ml_client

config = load_config()
ml_client = get_ml_client(config.azure)

print(f"✅ Successfully connected to workspace: {ml_client.workspace_name}")

## 10. Create Blob Container (if needed)

In [ ]:
# The storage module creates 'training-data' container automatically
# This cell verifies it exists

from azure.storage.blob import BlobServiceClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
account_url = f"https://{storage_account}.blob.core.windows.net"

blob_service_client = BlobServiceClient(account_url=account_url, credential=credential)

# List containers
containers = list(blob_service_client.list_containers())
print("Existing containers:")
for container in containers:
    print(f"  - {container.name}")

## Summary

✅ Infrastructure successfully provisioned!

**Created Resources:**
- Azure ML Workspace for model training
- Storage Account for training data
- Container Registry for container images
- Managed identities for secure access

## Next Steps

- **Notebook 03**: Provision compute cluster for training
- **Notebook 04**: Download Phi-4 model from Azure AI Foundry
- **Notebook 02**: Upload training data (if not done yet)

## Cleanup (Optional)

To destroy all resources (⚠️ careful!):
```bash
terraform destroy -var-file=environments/dev.tfvars
```